task 1

In [17]:
suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']
deck = [(rank, suit) for suit in suits for rank in ranks]

total_cards = len(deck)

red_cards = [card for card in deck if card[1] in ['Hearts', 'Diamonds']]
prob_red = len(red_cards) / total_cards

hearts = [card for card in red_cards if card[1] == 'Hearts']
prob_heart_given_red = len(hearts) / len(red_cards)

face_cards = [card for card in deck if card[0] in ['Jack', 'Queen', 'King']]

diamonds_face = [card for card in face_cards if card[1] == 'Diamonds']
prob_diamond_given_face = len(diamonds_face) / len(face_cards)

spade_or_queen = [card for card in face_cards if card[1] == 'Spades' or card[0] == 'Queen']
prob_spade_or_queen_given_face = len(spade_or_queen) / len(face_cards)

print("P(Red card):", prob_red)
print("P(Heart | Red):", prob_heart_given_red)
print("P(Diamond | Face card):", prob_diamond_given_face)
print("P(Spade OR Queen | Face card):", prob_spade_or_queen_given_face)

P(Red card): 0.5
P(Heart | Red): 0.5
P(Diamond | Face card): 0.25
P(Spade OR Queen | Face card): 0.5


task 2

In [5]:
%pip install pgmpy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 3.5 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 3.5 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 3.5 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 3.5 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 3.5 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 3.5 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 3.5 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 3.5 MB/s eta 0:00:01
   ------------------------------ --------- 1.8/2.4 MB 851.8 kB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 1.1 MB/s  0:00:02
   ----------------------

In [11]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([('intelligence', 'grade'),('StudyHours', 'grade'),('Difficulty', 'grade'),('grade', 'Pass')])

cpd_i = TabularCPD(variable='intelligence', variable_card=2,
                   values=[[0.7], [0.3]],
                   state_names={'intelligence': ['high', 'low']})

cpd_s = TabularCPD(variable='StudyHours', variable_card=2,
                   values=[[0.6], [0.4]],
                   state_names={'StudyHours': ['sufficient', 'Insufficient']})

cpd_d = TabularCPD(variable='Difficulty', variable_card=2,
                   values=[[0.4], [0.6]],
                   state_names={'Difficulty': ['hard', 'Easy']})

cpd_g = TabularCPD(
    variable='grade',
    variable_card=3,
    values=[[0.3, 0.6, 0.4, 0.8, 0.1, 0.4, 0.2, 0.6], [0.4, 0.3, 0.4, 0.15, 0.4, 0.4, 0.5, 0.3],[0.3, 0.1, 0.2, 0.05, 0.5, 0.2, 0.3, 0.1]],
    evidence=['intelligence', 'StudyHours', 'Difficulty'],
    evidence_card=[2, 2, 2],
    state_names={
        'grade': ['A', 'B', 'C'],
        'intelligence': ['high', 'low'],
        'StudyHours': ['sufficient', 'Insufficient'],
        'Difficulty': ['hard', 'Easy']
    }
)
cpd_p = TabularCPD(
    variable='Pass',
    variable_card=2,
    values=[[0.05, 0.2, 0.5],[0.95, 0.8, 0.5]],
    evidence=['grade'],
    evidence_card=[3],
    state_names={'Pass':['No','Yes'],'grade':['A','B','C']}
)
model.add_cpds(cpd_i, cpd_s, cpd_d, cpd_g, cpd_p)
assert model.check_model()

infer = VariableElimination(model)
q1 = infer.query(
    variables=['Pass'],
    evidence={'StudyHours': 'sufficient', 'Difficulty': 'hard'}
)
print("P(Pass | sufficient, hard):\n", q1)
q2 = infer.query(
    variables=['intelligence'],
    evidence={'Pass': 'Yes'}
)
print("P(intelligence | Pass=Yes):\n", q2)

P(Pass | sufficient, hard):
 +-----------+-------------+
| Pass      |   phi(Pass) |
+===========+=============+
| Pass(No)  |      0.2720 |
+-----------+-------------+
| Pass(Yes) |      0.7280 |
+-----------+-------------+
P(intelligence | Pass=Yes):
 +--------------------+---------------------+
| intelligence       |   phi(intelligence) |
+====================+=====================+
| intelligence(high) |              0.7163 |
+--------------------+---------------------+
| intelligence(low)  |              0.2837 |
+--------------------+---------------------+


task 3

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([('disease', 'fever'),('disease', 'cough'),
                                 ('disease', 'fatigue'),('disease', 'chills')])
cpd_disease = TabularCPD(variable='disease', variable_card=2,values=[[0.3], [0.7]],
                        state_names={'disease': ['flu', 'Cold']})

cpd_fever = TabularCPD(variable='fever', variable_card=2,values=[[0.1, 0.5], [0.9, 0.5]],evidence=['disease'], evidence_card=[2],
                       state_names={'fever': ['No', 'yes'], 'disease': ['flu', 'Cold']})

cpd_cough = TabularCPD(variable='cough', variable_card=2,
                       values=[[0.2, 0.4], [0.8, 0.6]], evidence=['disease'], evidence_card=[2],
                       state_names={'cough': ['No', 'yes'], 'disease': ['flu', 'Cold']})

cpd_fatigue = TabularCPD(variable='fatigue', variable_card=2,
                         values=[[0.3, 0.7], [0.7, 0.3]],evidence=['disease'], evidence_card=[2],state_names={'fatigue': ['No', 'yes'], 'disease': ['flu', 'Cold']})

cpd_chills = TabularCPD(variable='chills', variable_card=2,values=[[0.4, 0.6], [0.6, 0.4]],
                        evidence=['disease'], evidence_card=[2],
                        state_names={'chills': ['No', 'yes'], 'disease': ['flu', 'Cold']})
model.add_cpds(cpd_disease, cpd_fever, cpd_cough, cpd_fatigue, cpd_chills)
assert model.check_model()

infer = VariableElimination(model)
q1 = infer.query(variables=['disease'],
                 evidence={'fever': 'yes', 'cough': 'yes'})
print("P(disease | fever, cough):\n", q1)
q2 = infer.query(variables=['disease'],
                 evidence={'fever': 'yes', 'cough': 'yes', 'chills': 'yes'})
print("P(disease | fever, cough, chills):\n", q2)
q3 = infer.query(variables=['fatigue'],
                 evidence={'disease': 'flu'})
print("P(fatigue | flu):\n", q3)

P(disease | fever, cough):
 +---------------+----------------+
| disease       |   phi(disease) |
+===============+================+
| disease(flu)  |         0.5070 |
+---------------+----------------+
| disease(Cold) |         0.4930 |
+---------------+----------------+
P(disease | fever, cough, chills):
 +---------------+----------------+
| disease       |   phi(disease) |
+===============+================+
| disease(flu)  |         0.6067 |
+---------------+----------------+
| disease(Cold) |         0.3933 |
+---------------+----------------+
P(fatigue | flu):
 +--------------+----------------+
| fatigue      |   phi(fatigue) |
+==============+================+
| fatigue(No)  |         0.3000 |
+--------------+----------------+
| fatigue(yes) |         0.7000 |
+--------------+----------------+


task 4

In [ ]:
import numpy as np

states = ["Sunny", "Cloudy", "rainy"] #cloudy with a chance of meatballs :)) 

transition_matrix = np.array([
    [0.6, 0.3, 0.1],  
    [0.3, 0.4, 0.3],  
    [0.2, 0.3, 0.5]   
])

def simulate_weather(initial_state, days):
    current_state = initial_state
    sequence = [current_state]

    for _ in range(days):
        if current_state == "Sunny":
            next_state = np.random.choice(states, p=transition_matrix[0])
        elif current_state == "cloudy":
            next_state = np.random.choice(states, p=transition_matrix[1])
        else:
            next_state = np.random.choice(states, p=transition_matrix[2])

        sequence.append(next_state)
        current_state = next_state

    return sequence
days = 10
initial_state = "Sunny"
sequence = simulate_weather(initial_state, days)
print("weather sequence:")
print(" > phir > ".join(sequence))
trials = 10000
count = 0
for _ in range(trials):
    seq = simulate_weather(initial_state, days)
    rainy_days = seq.count("rainy")
    if rainy_days >= 3:
        count += 1
probability = count / trials
print("P(at least 3 rainy days in 10 days):", probability)

weather sequence:
Sunny > phir > Sunny > phir > Cloudy > phir > rainy > phir > Cloudy > phir > Cloudy > phir > rainy > phir > Cloudy > phir > rainy > phir > rainy > phir > rainy
P(at least 3 rainy days in 10 days): 0.6235
